# Single point calculation

To run a single point using aiida-mlip you need to define some inputs as AiiDA data types, to then pass them to the calculation.
First of all we need a structure on which to perform the calculations. It will be a NaCl structure that we define using ASE, or alternatively one can choose one of the structures in the folder `Structures`.
The input structure in aiida-mlip needs to be saved as a StructureData type:

In [1]:
from aiida import load_profile
load_profile()

Profile<uuid='a0bc4bdaef58416f840387aceea091a9' name='john'>

In [2]:
from aiida.orm import StructureData
from ase.build import bulk
from ase.io import read

#structure = StructureData(ase=read("Structures/qmof-ffeef76.cif"))
structure = StructureData(ase=bulk("NaCl", "rocksalt", 5.63))

Then we need to choose a model and architecture to be used for the calculation and save it as ModelData type, a specific data type of this plugin.
In this example we use MACE with a model that we download from this URL: "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model", and we save the file in the cache folder (default="~/.cache/mlips/"):


In [4]:
from aiida_mlip.data.model import ModelData
#uri = "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model"
#model = ModelData.from_uri(uri, architecture="mace_mp", cache_dir="mlips")

If we already have the model saved in some folder we can save it as:

In [5]:
model = ModelData.from_local("/mnt/87bd3caf-192d-48ed-9f63-c5afcf32b9ec/ML/janus_work/models/MACE-matpes-r2scan-omat-ft.model", architecture="mace")

Another parameter that we need to define as AiiDA type is the code. Assuming the code is saved as `janus` in the `localhost` computer, the code info that are needed can be loaded as follow:


In [6]:
from aiida.orm import load_code
code = load_code("janus@localhost")

The other inputs can be set up as AiiDA Str. There is a default for every input except the structure and code. This is a list of possible inputs:

In [7]:
from aiida.orm import Dict, Str

inputs = {
        "code": code,
        "model": model,
        "struct": structure,
        "arch": Str(model.architecture),
        "device": Str("cpu"),
        "calc_kwargs": Dict({"dispersion": True}),
        "metadata": {"options": {"resources": {"num_machines": 1}}},
    }

It's worth noting that the architecture is already defined within the model, accessible through the architecture property in the ModelData. Even if not explicitly provided as input, it will be automatically retrieved from the model.

The calculation must be set:

In [8]:
from aiida.plugins import CalculationFactory
phononCalc = CalculationFactory("mlip.ph")

In this case, since we are running a single point calculation the entry point for the calculation is `mlip.sp`.
Finally, run the calculation:


In [9]:
from aiida.engine import run_get_node
result, node = run_get_node(phononCalc, inputs)

02/03/2026 03:43:12 PM <211944> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
02/03/2026 03:43:12 PM <211944> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
02/03/2026 03:43:12 PM <211944> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.
02/03/2026 03:43:14 PM <211944> aiida.parser.PhononParser: [ERROR] Found files '['_scheduler-stderr.txt', '_scheduler-stdout.txt', 'aiida-stdout.txt']', expected to find '{'aiida-stdout.txt', 'aiida.log'}'
02/03/2026 03:43:14 PM <211944> aiida.orm.nodes.process.calculation.calcjob.CalcJobNode: [WARNING] output parser returned exit code<305>: Some output files missing or cannot be read


`result` is a dictionary of the available results obtained from the calculation, while node contains the infor on the node where the calculation is run:


In [10]:
print(result)
print(node)

{'remote_folder': <RemoteData: uuid: a7e64675-d724-4b86-bdb8-b239910da747 (pk: 1965)>, 'retrieved': <FolderData: uuid: d1c15eb4-bead-4d89-83ed-e66f32cd5262 (pk: 1966)>}
uuid: 7136cc32-0f42-457b-beea-17733ec0f554 (pk: 1964) (aiida.calculations:mlip.ph)


We can check if the calculation finished with errors. If everything worked the exit code should be 0

In [ ]:
if node.is_finished_ok:
     print(f"Caculation is finished without errors with exit status {node.exit_status}")
else:
     print(f"Some errors occurred with exit status {node.exit_status}") 

If more information are needed on specific outputs they can be called like:

In [ ]:
print(result["results_dict"].get_dict())

Let's say we want the energy we can access it via the result or the node variable.
(when this is run in the terminal the auto-completion should help, but the idea is that the results_dict is one of the outputs which contains the main info on the calculation)

In [ ]:
print(f"Energy: {result['results_dict'].get_dict()['info']['mace_mp_d3_energy']}")
print(f"Energy: {node.outputs.results_dict.get_dict()['info']['mace_mp_d3_energy']}")
print(node.outputs.results_dict.get_dict()['cell'][0][1])

Through the command line we can see the processes that are run

In [ ]:
! verdi process list -a

And see the results we are interested in. Substitute the number with the PK number of your calculation

In [ ]:
! verdi calcjob res pk

We can also see the inputs and outputs of the calculation

In [ ]:
! verdi node show pk  